<a href="https://colab.research.google.com/github/dagmaros27/AIMS_Notebooks/blob/main/TinyLORA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("wake up")

wake up


# TinyLORA
Reporoducing the paper "Learning to Reason in 13 Parameters
" by Morris et al. [Link](https://arxiv.org/pdf/2602.04118)

In [ ]:
!pip install -q transformers datasets trl torch peft accelerate

## Load Model

## Load Dataset
we used GSM8K for maths

In [53]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True,
                                             device_map="auto", dtype=torch.bfloat16)

model.gradient_checkpointing_enable()

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [54]:
from datasets import load_dataset

dataset_name = "openai/gsm8k"

dataset = load_dataset(dataset_name, "main")
print(f"length of dataset: {len(dataset['train'])}")

length of dataset: 7473


In [55]:
import torch
import torch.nn as nn
import math

class TinyLORA(nn.Module):
    def __init__(self, original_linear: nn.Linear, shared_v, r=2, u=1):
        super().__init__()
        self.in_features = original_linear.in_features
        self.out_features = original_linear.out_features
        self.u = u

        device = original_linear.weight.device

        # freezing the weights and bias
        self.weight = original_linear.weight
        self.bias = original_linear.bias
        self.weight.requires_grad = False
        if self.bias is not None:
            self.bias.requires_grad = False

        # perform SVD on the original layer
        W_float = self.weight.float().cpu()
        U, S, V = torch.linalg.svd(W_float, full_matrices=False)

        #turncate to rank r
        self.U = nn.Parameter(U[:, :r].to(device=device, dtype=self.weight.dtype), requires_grad=False)
        self.S = nn.Parameter(torch.diag(S[:r]).to(device=device, dtype=self.weight.dtype), requires_grad=False)
        self.V = nn.Parameter(V[:r, :].to(device=device, dtype=self.weight.dtype), requires_grad=False)

        # create a random matrixes P_i
        self.P = nn.Parameter(
            torch.randn(u, r, r, dtype=self.weight.dtype, device=device),
            requires_grad=False
        )

        #store shared vector
        self.shared_v = shared_v.to(device)

    def forward(self, x):
        base_output = nn.functional.linear(x, self.weight, self.bias)

        R = torch.zeros_like(self.P[0])
        for i in range(self.u):
            R += self.shared_v[i].to(self.P.device) * self.P[i]

        W_delta = self.U @ self.S @ R @ self.V
        lora = nn.functional.linear(x, W_delta)

        return base_output + lora

In [56]:
u = 4

shared_v = nn.Parameter(torch.zeros(u))
optimizer = torch.optim.Adam([shared_v], lr=1e-3)


# generate answers for the prompt
def generate_response(model, tokenizer, prompt, n_samples=8):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids
    prompt_length = input_ids.shape[1]

    outputs = model.generate(
        input_ids=input_ids,
        num_return_sequences=n_samples,
        temperature=0.8,
        do_sample=True,
    )

    responses = [tokenizer.decode(out[prompt_length:], skip_special_tokens=True) for out in outputs]
    return responses

def compute_logprob(model, tokenizer, prompt, response):
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids
    prompt_length = prompt_ids.shape[1]

    text = prompt + response
    inputs = tokenizer(text, return_tensors="pt")
    input_ids = inputs["input_ids"]

    outputs = model(input_ids)
    logits = outputs.logits
    log_probs = nn.functional.log_softmax(logits, dim=-1)

    target_ids = input_ids[:, 1:]
    log_probs = log_probs[:, :-1, :]

    token_logprobs = log_probs.gather(
        -1,
        target_ids.unsqueeze(-1)
    ).squeeze(-1)

    return token_logprobs[:, prompt_length - 1:].sum()

import re

# both extract_number and reward_fn depend on the model type
def extract_number(text):
    # prefer #### (GSM8K format)
    if '####' in text:
        text = text.split('####')[-1]
    # prefer \boxed{X}
    elif '\\boxed{' in text:
        boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
        if boxed:
            text = boxed[0]
    else:
        # just take the first 300 chars before hallucinations start
        text = text[:300]

    text = text.replace(',', '')
    match = re.findall(r"-?\d+(?:\.\d+)?", text)
    return float(match[-1]) if match else None

def reward_fn(prediction, answer):
    try:
        pred = extract_number(prediction)
        # this is because gsm8k show final answer after '####'
        true_ans_str = answer.split('####')[-1]
        true_ans = extract_number(true_ans_str)
        return 1.0 if pred == true_ans else 0.0
    except:
        return 0.0

def grpo():
    for batch in dataset['train']:
        prompt = batch['question']
        answer = batch['answer']

        responses = generate_response(model, tokenizer, prompt)

        rewards = []
        logprobs = []

        for response in responses:
            rewards.append(reward_fn(response, answer))
            logprobs.append(compute_logprob(model, tokenizer, prompt, response))

        rewards = torch.tensor(rewards)
        logprobs = torch.stack(logprobs)

        advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-8)

        loss = (-logprobs * advantages).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [57]:
import tqdm

import torch
from tqdm import tqdm

def evaluate_model(model, tokenizer, dataset, split="test", num_samples=100):
    # put model on evaluation mode
    model.eval()

    correct = 0
    total = 0


    test_data = dataset[split].select(range(num_samples))

    print(f"Starting evaluation on {num_samples} problems...")

    for batch in tqdm(test_data, desc="Evaluating"):
        prompt = batch['question']
        ground_truth = batch['answer']

        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
        prompt_length = input_ids.shape[1]


        with torch.no_grad():
         outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_type_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        response = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)


        score = reward_fn(response, ground_truth)

        if score == 1.0:
            correct += 1
        total += 1

    accuracy = (correct / total) * 100

    print(f"\nEvaluation Complete!")
    print(f"Accuracy: {accuracy:.2f}% ({correct}/{total} correct)")
    return accuracy

In [58]:

baseline_acc = evaluate_model(model, tokenizer,OAS dataset, num_samples=10)
print(f"Baseline: {baseline_acc:.2f}%")

Starting evaluation on 10 problems...


Evaluating: 100%|██████████| 10/10 [04:57<00:00, 29.70s/it]


Evaluation Complete!
Accuracy: 70.00% (7/10 correct)
Baseline: 70.00%


In [ ]:
import time
import matplotlib.pyplot as plt

# inject tinylora into the model
def inject_tinylora(model, shared_v, r=2, u=4,
                    target_modules=("q_proj", "v_proj", "k_proj", "o_proj",
                                    "up_proj", "down_proj", "gate_proj")):
    for name, module in list(model.named_modules()):
        parent_name, child_name = name.rsplit(".", 1) if "." in name else ("", name)
        if child_name in target_modules and isinstance(module, nn.Linear):
            parent = model if parent_name == "" else dict(model.named_modules())[parent_name]
            setattr(parent, child_name, TinyLORA(module, shared_v, r=r, u=u))
    return model

model = inject_tinylora(model, shared_v, r=2, u=u)

for param in model.parameters():
    param.requires_grad = False
shared_v.requires_grad = True

# baseline eval

baseline_acc = evaluate_model(model, tokenizer, dataset)
print(f"Baseline: {baseline_acc:.2f}%")

# training loop

history = {"loss": [], "reward": [], "step_time": []}

for batch in tqdm(dataset["train"], desc="GRPO"):
    t0 = time.time()

    responses = generate_response(model, tokenizer, batch["question"])
    rewards   = torch.tensor([reward_fn(r, batch["answer"]) for r in responses])
    logprobs  = torch.stack([compute_logprob(model, tokenizer, batch["question"], r) for r in responses])

    adv  = (rewards - rewards.mean()) / (rewards.std() + 1e-8)
    loss = (-logprobs * adv).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    step_time = time.time() - t0
    history["loss"].append(loss.item())
    history["reward"].append(rewards.mean().item())
    history["step_time"].append(step_time)

# post-training eval
trained_acc = evaluate_model(model, tokenizer, dataset)
print(f"Trained:  {trained_acc:.2f}%  (Δ {trained_acc - baseline_acc:+.2f} pp)")


fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].plot(history["loss"])
axes[0].set_title("Loss")
axes[0].set_xlabel("Step")

axes[1].plot(history["reward"])
axes[1].set_title("Mean Reward")
axes[1].set_xlabel("Step")

axes[2].plot(history["step_time"])
axes[2].set_title("Step Time (s)")
axes[2].set_xlabel("Step")

axes[3].bar(["Baseline", "Trained"], [baseline_acc, trained_acc], color=["#ff7c7c", "#7cffb2"])
for i, v in enumerate([baseline_acc, trained_acc]):
    axes[3].text(i, v + 0.3, f"{v:.1f}%", ha="center", fontweight="bold")
axes[3].set_title("GSM8K Accuracy")
axes[3].set_ylabel("%")

plt.tight_layout()
plt.savefig("tinylora_results.png", dpi=150)
plt.show()

Starting evaluation on 100 problems...


Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]


RuntimeError: Expected all tensors to be on the same device, but got mat2 is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_mm)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True,
                                             device_map="auto", dtype=torch.bfloat16)
model.gradient_checkpointing_enable()

from datasets import load_dataset
dataset_name = "openai/gsm8k"
dataset = load_dataset(dataset_name, "main")
print(f"length of dataset: {len(dataset['train'])}")

import torch
import torch.nn as nn
import math

class TinyLORA(nn.Module):
    def __init__(self, original_linear: nn.Linear, shared_v, r=2, u=1):
        super().__init__()
        self.in_features = original_linear.in_features
        self.out_features = original_linear.out_features
        self.u = u

        self.weight = original_linear.weight
        self.bias = original_linear.bias
        self.weight.requires_grad = False
        if self.bias is not None:
            self.bias.requires_grad = False

        W_float = self.weight.float()
        U,S,V = torch.linalg.svd(W_float, full_matrices=False)

        self.U = nn.Parameter(U[:, :r].to(self.weight.dtype), requires_grad=False)
        self.S = nn.Parameter(torch.diag(S[:r].to(self.weight.dtype)), requires_grad=False)
        self.V = nn.Parameter(V[:r, :].to(self.weight.dtype), requires_grad=False)

        self.P = nn.Parameter(torch.randn(u,r,r, dtype=self.weight.dtype, device=self.weight.device), requires_grad=False)

        self.shared_v = shared_v.to(self.weight.device)

    def forward(self, x):
        base_output = nn.functional.linear(x, self.weight, self.bias)

        R = torch.zeros_like(self.P[0])
        for i in range(self.u):
            R += self.shared_v[i].to(self.P.device) * self.P[i]

        W_delta = self.U @ self.S @ R @ self.V
        lora = nn.functional.linear(x, W_delta)
        return base_output + lora

u = 4
shared_v = nn.Parameter(torch.zeros(u))
optimizer = torch.optim.Adam([shared_v], lr=1e-3)

def generate_response(model, tokenizer, prompt, n_samples=8):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    prompt_length = input_ids.shape[1]

    outputs = model.generate(
        input_ids=input_ids,
        num_return_sequences=n_samples,
        temperature=0.8,
        do_sample=True,
    )

    responses = [tokenizer.decode(out[prompt_length:], skip_special_tokens=True) for out in outputs]
    return responses

def compute_logprob(model, tokenizer, prompt, response):
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids
    prompt_length = prompt_ids.shape[1]

    text = prompt + response
    inputs = tokenizer(text, return_tensors="pt")
    input_ids = inputs["input_ids"].to(model.device)

    outputs = model(input_ids)
    logits = outputs.logits
    log_probs = nn.functional.log_softmax(logits, dim=-1)

    target_ids = input_ids[:, 1:]
    log_probs = log_probs[:, :-1, :]

    token_logprobs = log_probs.gather(
        -1,
        target_ids.unsqueeze(-1)
    ).squeeze(-1)

    return token_logprobs[:, prompt_length - 1:].sum()

import re

def extract_number(text):
    text = text.replace(',', '')
    match = re.findall(r"-?\d+(?:\.\d+)?", text)
    if not match:
        return None
    return float(match[-1])

def reward_fn(prediction, answer):
    try:
        pred = extract_number(prediction)
        true_ans_str = answer.split('####')[-1]
        true_ans = extract_number(true_ans_str)
        return 1.0 if pred == true_ans else 0.0
    except:
        return 0.0

def grpo():
    for batch in dataset['train']:
        prompt = batch['question']
        answer = batch['answer']

        responses = generate_response(model, tokenizer, prompt)

        rewards = []
        logprobs = []
        for response in responses:
            rewards.append(reward_fn(response, answer))
            logprobs.append(compute_logprob(model, tokenizer, prompt, response))

        # FIX: Send the rewards tensor to the GPU and ensure it's a float for the math below
        rewards = torch.tensor(rewards, dtype=torch.float32, device=model.device)
        logprobs = torch.stack(logprobs)

        advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-8)
        loss = (-logprobs * advantages).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

import tqdm
import torch
from tqdm import tqdm

def evaluate_model(model, tokenizer, dataset, split="test", num_samples=100):
    model.eval()
    correct = 0
    total = 0
    test_data = dataset[split].select(range(num_samples))

    print(f"Starting evaluation on {num_samples} problems...")

    for batch in tqdm(test_data, desc="Evaluating"):
        prompt = batch['question']
        ground_truth = batch['answer']

        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
        prompt_length = input_ids.shape[1]

        with torch.no_grad():
            outputs = model.generate(
                input_ids=input_ids,
                max_new_tokens=512,
                temperature=0.0,
                do_sample=False
            )

        response = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)
        score = reward_fn(response, ground_truth)

        if score == 1.0:
            correct += 1
        total += 1

    accuracy = (correct / total) * 100
    print(f"\nEvaluation Complete!")
    print(f"Accuracy: {accuracy:.2f}% ({correct}/{total} correct)")
    return accuracy

import time
import matplotlib.pyplot as plt

def inject_tinylora(model, shared_v, r=2, u=4,
                    target_modules=("q_proj", "v_proj", "k_proj", "o_proj",
                                    "up_proj", "down_proj", "gate_proj")):
    for name, module in list(model.named_modules()):
        parent_name, child_name = name.rsplit(".", 1) if "." in name else ("", name)
        if child_name in target_modules and isinstance(module, nn.Linear):
            parent = model if parent_name == "" else dict(model.named_modules())[parent_name]
            setattr(parent, child_name, TinyLORA(module, shared_v, r=r, u=u))
    return model

model = inject_tinylora(model, shared_v, r=2, u=u)

for param in model.parameters():
    param.requires_grad = False
shared_v.requires_grad = True

# print("Evaluating baseline")
# baseline_acc = evaluate_model(model, tokenizer, dataset)
# print(f"Baseline: {baseline_acc:.2f}%")

history = {"loss": [], "reward": [], "step_time": []}
for batch in tqdm(dataset["train"], desc="GRPO"):
    t0 = time.time()
    responses = generate_response(model, tokenizer, batch["question"])
    rewards   = torch.tensor([reward_fn(r, batch["answer"]) for r in responses])
    logprobs  = torch.stack([compute_logprob(model, tokenizer, batch["question"], r) for r in responses])

    adv  = (rewards - rewards.mean()) / (rewards.std() + 1e-8)
    loss = (-logprobs * adv).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    step_time = time.time() - t0
    history["loss"].append(loss.item())
    history["reward"].append(rewards.mean().item())
    history["step_time"].append(step_time)

print("evaluating TinyLORA")
trained_acc = evaluate_model(model, tokenizer, dataset)
print(f"Trained:  {trained_acc:.2f}%  (Δ {trained_acc - baseline_acc:+.2f} pp)")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].plot(history["loss"])
axes[0].set_title("Loss")
axes[0].set_xlabel("Step")

axes[1].plot(history["reward"])
axes[1].set_title("Mean Reward")
axes[1].set_xlabel("Step")

axes[2].plot(history["step_time"])
axes[2].set_title("Step Time (s)")
axes[2].set_xlabel("Step")

axes[3].bar(["Baseline", "Trained"], [baseline_acc, trained_acc], color=["#ff7c7c", "#7cffb2"])
for i, v in enumerate([baseline_acc, trained_acc]):
    axes[3].text(i, v + 0.3, f"{v:.1f}%", ha="center", fontweight="bold")
axes[3].set_title("GSM8K Accuracy")
axes[3].set_ylabel("%")

plt.tight_layout()
plt.savefig("tinylora_results.png", dpi=150)
plt.show()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

length of dataset: 7473


GRPO:   0%|          | 0/7473 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
GRPO:   0%|          | 0/7473 [01:18<?, ?it/s]


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!